# Week 16 – Research Assistant Agent with Tool Calls 16.3 Graded Mini Project
### Submitted by: Harley Dangan
### Date: 26 November 2026
### Submitted to: The Asian Institute of Management
### License: GNU General Public License v3.0 (GPL-3.0-or-later) - see LICENSE



## Load Environment Variables

This Jupyter notebook assumes that you have the environment variables configured. See .venv.example for reference.

**IMPORTANT NOTES:**

**langchain_ollama is used in this notebook. This notebook will only work with Ollama, Llama.cpp, or any similar inference engine.**

**The following command needs to be run in order to install the jailbreak guardrail models:**
**uv run python -c "from guardrails_ai.detect_jailbreak import DetectJailbreak; DetectJailbreak(on_fail='noop')"**

**Laya, a System One model, is also used in this project. It also needs to be configured in the environment variables. Host Laya locally or use the online API service.**

In [ ]:
from dotenv import load_dotenv

# Read .env from the notebook's directory. Variables already set in the
# environment take precedence over the file.
load_dotenv()

import os

base_url = os.environ.get("BASE_URL")
llm= os.environ.get("LLM")
api_key = os.environ.get("OPENAI_API_KEY") # Actually, any string will work here since I used Ollama!

print("LLM configuration:")
if not base_url or not llm or not api_key:
    raise ValueError("Missing required environment variables: BASE_URL, LLM, API_KEY, or TAVILY_API_KEY")
print(f"Base URL: {base_url}")
print(f"LLM: {llm}")
print(f"API Key: {"set" if api_key else "not set"}")

print("\nLaya configuration:")
laya_base_url = os.environ.get("LAYA_BASE_URL")
laya_api_key = os.environ.get("LAYA_API_KEY")
laya_llm = os.environ.get("LLM")
if not laya_base_url or not laya_api_key or not laya_llm:
    raise ValueError("Missing required environment variables: LAYA_BASE_URL, LAYA_API_KEY, or LLM")
print(f"Laya Base URL: {laya_base_url}")
print(f"Laya LLM: {laya_llm}")
print(f"Laya API Key: {"set" if laya_api_key else "not set"}")

tavily_api_key = os.environ.get("TAVILY_API_KEY")
if not tavily_api_key:
    raise ValueError("Missing required environment variable: TAVILY_API_KEY")

print("\nLangsmith configuration:")
langsmith_api_key = os.environ.get("LANGSMITH_API_KEY")
if not langsmith_api_key:
    raise ValueError("Missing required environment variable: LANGSMITH_API_KEY")
print(f"LangSmith API Key: {"set" if langsmith_api_key else "not set"}")

## Initialize LLM

Creates the `ChatOllama` client from the `LLM` and `BASE_URL` environment
variables and defines the system prompt that governs the agent's research,
tool use, and citation behaviour.

**NOTE:** Assumes a local model running in Ollama.

In [ ]:
from langchain_ollama import ChatOllama

# Initialize the ChatOllama model with the specified parameters
llm = ChatOllama(
    model=llm,
    temperature=0.3,
    base_url=base_url,
)

# Defining the system prompt for the model
system_prompt = """You are an expert Research Assistant Agent. Your core objective is to deliver comprehensive, highly accurate, and fact-verified research on any given topic. You have access to tools for searching the web and retrieving full webpage content.

### OPERATIONAL MANDATES
1. FACTUAL GROUNDING: Rely exclusively on verifiable information retrieved from your tools. If a fact cannot be sourced from your tool outputs, explicitly state that it is unavailable. Never assume, extrapolate, or hallucinate details.
2. LATENT SEARCH QUERIES: When an initial tool result is ambiguous, outdated, or incomplete, proactively formulate distinct follow-up queries or browse primary source URLs to fill knowledge gaps.
3. CONTRADICTION RESOLUTION: If different sources present conflicting numbers, dates, or accounts, outline both perspectives objectively. Do not pick a side unless one source is demonstrably more credible.
4. SYNTHESIS OVER SUMMARIZATION: Do not simply list summaries of search results. Synthesize the findings into a cohesive narrative structured with clear headers, logical groupings, and thematic continuity.

### TOOL-CALLING GUIDELINES
- Query Design: Keep search tool queries targeted and optimized. Avoid passing long conversational sentences to search engines. Use precise keywords, dates, or entity names.
- Deep Dive: If a search snippet provides a promising lead but lacks critical context, use the browser tool to read the full page before finalizing your synthesis.
- Temporal Context: Always keep the current year and context in mind to evaluate whether information is current or obsolete.

### OUTPUT STRUCTURE & STYLE
- Formality: Maintain a neutral, professional, and academic tone. Avoid conversational filler, personal pronouns, or fluff.
- Scannability: Use descriptive Markdown headers, bold key entities/metrics, and utilize bullet points for non-sequential data points.
- Citation: Always attach functional anchors or clear indicators mapping back to the specific source domain or URL retrieved by your tools."""



## Initialize Tools

The retrieval and citation functions are declared with LangChain's `@tool`
decorator. `tools_schema` is generated from those tools, so the JSON schema
cannot drift from the function signatures.


In [ ]:
import json
import os
import re
from urllib.parse import urlparse

import requests

from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from tavily import TavilyClient
import arxiv

# Instantiate the Tavily client once so the LangGraph nodes can call it.
tavily_client = TavilyClient(api_key=tavily_api_key)

# ---------------------------------------------------------------------------
# Tavily mock switch, driven by .env
#   TAVILY_MOCK=1 (default) -> nodes return canned results, no API credits used
#   TAVILY_MOCK=0           -> nodes call the real Tavily API
# ---------------------------------------------------------------------------
_FALSEY = {"0", "false", "no", "off", ""}
USE_MOCK_TAVILY = os.getenv("TAVILY_MOCK", "1").strip().lower() not in _FALSEY

print(f"Tavily search mode: {'MOCK (no API calls)' if USE_MOCK_TAVILY else 'LIVE (Tavily API)'}")

# Canned payload shaped like TavilyClient.search(). The `query` field is
# overwritten at call time so outputs still show what was actually asked.
MOCK_TAVILY_RESPONSE = {
    "mock": True,
    "answer": (
        "Mock summary: recent retrieval-augmented generation work focuses on better "
        "chunking, hybrid retrieval with reranking, and measuring how well answers "
        "stay grounded in the retrieved sources."
    ),
    "response_time": 0.42,
    "results": [
        {
            "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
            "url": "https://example.com/mock/rag-knowledge-intensive-nlp",
            "content": (
                "Mock document. Pairs a dense retriever with a seq2seq generator and reports "
                "gains on open-domain question answering and fact verification when the "
                "retriever is trained jointly with the generator."
            ),
            "score": 0.94,
        },
        {
            "title": "Hybrid Retrieval and Reranking for Grounded Generation",
            "url": "https://example.com/mock/hybrid-retrieval-reranking",
            "content": (
                "Mock document. Combines BM25 and dense vectors, then applies a cross-encoder "
                "reranker; the reranking stage accounts for most of the accuracy improvement "
                "on long-form answers."
            ),
            "score": 0.89,
        },
        {
            "title": "Evaluating Faithfulness in Retrieval-Augmented Systems",
            "url": "https://example.com/mock/evaluating-faithfulness",
            "content": (
                "Mock document. Introduces a benchmark that separates retrieval recall from "
                "citation accuracy, finding that models often cite plausible but unretrieved "
                "passages."
            ),
            "score": 0.83,
        },
        {
            "title": "Chunking Strategies for Scientific Literature Retrieval",
            "url": "https://example.com/mock/chunking-scientific-literature",
            "content": (
                "Mock document. Compares fixed-window, recursive and semantic chunking on arXiv "
                "corpora; semantic chunking improves answer completeness but increases index "
                "size substantially."
            ),
            "score": 0.77,
        },
    ],
}


def mock_tavily_search(query: str, max_results: int = 5) -> dict:
    """Return canned results in the same shape as TavilyClient.search()."""
    payload = dict(MOCK_TAVILY_RESPONSE)
    payload["query"] = query
    payload["results"] = MOCK_TAVILY_RESPONSE["results"][:max_results]
    return payload


class SearchAPIError(RuntimeError):
    """A retrieval API failed; the message is safe to show to the user."""

    MESSAGES = {
        "Tavily": (
            "The web search is unavailable right now. The Tavily API returned "
            "an error; check the API key or try again shortly."
        ),
        "arXiv": (
            "The arXiv search is unavailable right now. The arXiv API is "
            "rate-limiting requests or temporarily down; wait a minute and try again."
        ),
    }

    def __init__(self, source: str, detail: str = ""):
        self.source = source
        self.detail = detail
        super().__init__(self.MESSAGES.get(source, f"The {source} search failed."))


def run_tavily_search(query: str, max_results: int = 5) -> dict:
    """Single dispatch point: mock data or the real API, depending on the switch."""
    if USE_MOCK_TAVILY:
        return mock_tavily_search(query=query, max_results=max_results)
    try:
        return tavily_client.search(query=query, max_results=max_results)
    except Exception as exc:
        raise SearchAPIError("Tavily", str(exc)) from exc


@tool
def tavily_search(query: str) -> str:
    """Search the live web for current facts, news, and technical data."""
    result = run_tavily_search(query=query)
    return json.dumps(result) if isinstance(result, (dict, list)) else str(result)


# Arxiv Search Function
class _ArxivSession(requests.Session):
    """requests session with a default timeout, so a stalled arXiv call cannot hang the notebook."""

    def request(self, *args, **kwargs):
        kwargs.setdefault("timeout", 20)
        return super().request(*args, **kwargs)


# One reusable arXiv client. In arxiv>=4 the client (not the Search object)
# exposes .results(), so searches must go through arxiv_client. arXiv throttles
# clients and occasionally answers 503, so a couple of retries are built in.
arxiv_client = arxiv.Client(num_retries=2, delay_seconds=3.0)
arxiv_client._session = _ArxivSession()


@tool
def arxiv_search(query: str, max_results: int = 5) -> str:
    """Search arXiv for academic papers, preprints, and research articles."""
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.Relevance
    )

    try:
        papers = list(arxiv_client.results(search))
    except Exception as exc:
        raise SearchAPIError("arXiv", str(exc)) from exc

    results = []
    for result in papers:
        paper_info = (
            f"Title: {result.title}\n"
            f"Authors: {', '.join(author.name for author in result.authors)}\n"
            f"Published: {result.published.strftime('%Y-%m-%d')}\n"
            f"URL: {result.pdf_url}\n"
            f"Abstract: {result.summary}\n"
            f"{'-'*40}"
        )
        results.append(paper_info)
        
    return "\n".join(results) if results else "No papers found for this query."

# APA Citation Function
# Converts the raw output of tavily_search_node or arxiv_search_node into
# APA 7th edition references. The input format is detected automatically, and
# the source can also be forced with source="tavily" or source="arxiv".

_ARXIV_URL_RE = re.compile(r"arxiv\.org/(?:pdf|abs)/(.+?)(?:v\d+)?/?$")
_YEAR_RE = re.compile(r"\b(1[5-9]\d{2}|20\d{2})\b")


def _apa_author(full_name: str) -> str:
    """'Jane Q. Doe' -> 'Doe, J. Q.'"""
    parts = [part for part in re.split(r"\s+", full_name.strip()) if part]
    if not parts:
        return ""
    if len(parts) == 1:
        return parts[0]
    family = parts[-1]
    initials = " ".join(f"{part[0].upper()}." for part in parts[:-1])
    return f"{family}, {initials}"


def _apa_author_list(authors: str) -> str:
    """'Jane Doe, John Smith' -> 'Doe, J., & Smith, J.'"""
    people = [name for name in (_apa_author(a) for a in authors.split(",")) if name]
    if not people:
        return ""
    if len(people) == 1:
        return people[0]
    if len(people) == 2:
        return f"{people[0]}, & {people[1]}"
    return ", ".join(people[:-1]) + f", & {people[-1]}"


def _apa_year(text: str) -> str:
    match = _YEAR_RE.search(text or "")
    return match.group(1) if match else "n.d."


def _apa_web_reference(item: dict) -> str:
    """Tavily result -> APA reference for a web page."""
    title = (item.get("title") or "").strip().rstrip(".")
    url = (item.get("url") or "").strip()
    site = urlparse(url).netloc.removeprefix("www.") or "Web"
    year = _apa_year(item.get("published_date") or item.get("published") or "")
    # With no author, the page title takes the author position.
    title_part = f"{title}. " if title else ""
    return f"{title_part}({year}). {site}. {url}"


def _apa_preprint_reference(block: str) -> str:
    """One arXiv entry -> APA reference for a preprint."""
    head = block.split("Abstract:", 1)[0]
    fields = {}
    for line in head.splitlines():
        key, separator, value = line.partition(":")
        if separator:
            fields[key.strip().lower()] = value.strip()

    title = fields.get("title", "").strip().rstrip(".")
    authors = _apa_author_list(fields.get("authors", ""))
    year = _apa_year(fields.get("published", ""))
    url = fields.get("url", "")

    match = _ARXIV_URL_RE.search(url)
    link = f"https://doi.org/10.48550/arXiv.{match.group(1)}" if match else url

    if authors and not authors.endswith("."):
        authors += "."
    author_part = f"{authors} " if authors else ""
    return f"{author_part}({year}). {title}. arXiv. {link}"


@tool
def apa_convert(payload: str, source: str = "auto") -> str:
    """
    Convert search output from tavily_search_node or arxiv_search_node into
    APA style citations, one reference per paragraph.

    payload: raw ToolMessage content (a JSON string for Tavily, formatted text
             for arXiv), or the already-parsed dict/list.
    source:  "auto" (default), "tavily", or "arxiv".
    """
    if isinstance(payload, (dict, list)):
        data = payload
        text = json.dumps(payload)
    else:
        data = None
        text = str(payload)

    if isinstance(payload, str) and "No papers found" in payload:
        return "No citations to convert: the arXiv search returned no papers."

    if source == "auto":
        stripped = text.lstrip()
        if stripped.startswith(("{", "[")):
            source = "tavily"
        elif "Title:" in text and "Authors:" in text:
            source = "arxiv"
        else:
            raise ValueError(
                "Unrecognised search output: expected Tavily JSON or arXiv text."
            )

    if source == "tavily":
        if data is None:
            data = json.loads(text)
        results = data.get("results", []) if isinstance(data, dict) else data
        references = [
            _apa_web_reference(item)
            for item in results
            if item.get("url") or item.get("title")
        ]
    elif source == "arxiv":
        if "Title:" not in text or "Authors:" not in text:
            raise ValueError("Input does not look like arXiv search output.")
        blocks = [block.strip() for block in text.split("-" * 40) if block.strip()]
        references = [_apa_preprint_reference(block) for block in blocks]
    else:
        raise ValueError(f"Unknown source {source!r}; use 'auto', 'tavily' or 'arxiv'.")

    # dict.fromkeys removes duplicates while preserving the original order.
    references = [ref for ref in dict.fromkeys(references) if ref]
    if not references:
        return "No citations to convert."
    return "\n\n".join(references)


# Map tool names to the LangChain tools created with the @tool decorator.
TOOLS = (tavily_search, arxiv_search, apa_convert)
available_tools = {tool.name: tool for tool in TOOLS}
tools_schema = [convert_to_openai_tool(tool) for tool in TOOLS]


## Create LangGraph State Class

`AgentState` is the single state object every node reads from and writes to.
Declaring it as a `TypedDict` gives LangGraph one schema for the whole run,
and LangGraph Studio uses it to show fields such as the route, synthesis, and
the Laya evaluation results in the state panel.


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph.message import add_messages

# State is one shared object every node reads from and writes to. 
# It is a TypedDict that defines the structure of the state object.
class AgentState(TypedDict):
    question: str
    answer: str
    threat_detected: bool
    api_error: str
    citation_format: str
    citation_scores: dict
    citation_reason: str
    research_route: str
    research_scores: dict
    research_reason: str
    synthesis: str
    eval_score: float | None
    eval_report: str
    eval_details: dict
    messages: Annotated[list, add_messages]

## Define LangGraph Search Node Functions

`run_tavily_search` and `arxiv_search` raise `SearchAPIError` when their API
fails (for example arXiv's HTTP 429 rate limit). The search node catches it,
stores a user-readable message in `state["api_error"]`, and the graph routes
straight to `display_results_node`, so the run ends with an explanation instead
of a raw traceback or a misleading "no results" synthesis.


In [ ]:
from typing import Dict, List

from langchain_core.messages import AIMessage, ToolMessage

# The node signatures below use `State`, an alias for the AgentState TypedDict
# defined in the "Create LangGraph State Class" section.
State = AgentState

# Both search nodes share the same failure handling: a SearchAPIError from the
# tool layer is written to state instead of crashing the run, and the graph's
# conditional edge sends the user-readable message straight to the display node.
def _search_error_update(error: SearchAPIError) -> dict:
    """Return the state update for a failed retrieval API call."""
    # Keep the technical detail in the run log; state only carries the
    # user-readable message that display_results_node prints.
    print(f"[search] {error.source} API error: {error.detail}")
    message = str(error)
    return {
        "api_error": message,
        "answer": message,
        "messages": [AIMessage(content=message)],
    }


# Web Search Result node: This node takes the current state, which includes a question, and uses the LLM to generate an answer. It returns a dictionary containing the answer.
def tavily_search_node(state: State) -> dict:
    """
    LangGraph Node that executes the Tavily search tool call.
    """
    latest_message = state["messages"][-1]
    tool_messages = []
    
    if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
        for tool_call in latest_message.tool_calls:
            if tool_call["name"] == "tavily_search":
                try:
                    # Execute Tavily search (mock or live, see USE_MOCK_TAVILY)
                    query = tool_call["args"].get("query")
                    tool_output = tavily_search.invoke({"query": query})
                    
                except SearchAPIError as error:
                    return _search_error_update(error)
                
                tool_messages.append(
                    ToolMessage(
                        content=str(tool_output),
                        tool_call_id=tool_call["id"],
                        name=tool_call["name"]
                    )
                )
                
    return {"messages": tool_messages}

# Arxiv Search node: This node takes the current state, which includes a question, 
# and uses the LLM to generate an answer. It returns a dictionary containing the answer.
def arxiv_search_node(state: State) -> dict:
    """
    LangGraph Node that executes the arXiv search tool call.
    """
    latest_message = state["messages"][-1]
    tool_messages = []
    
    if hasattr(latest_message, "tool_calls") and latest_message.tool_calls:
        for tool_call in latest_message.tool_calls:
            if tool_call["name"] == "arxiv_search":
                try:
                    # Extract arguments passed by the LLM
                    args = tool_call["args"]
                    query = args.get("query")
                    max_results = args.get("max_results", 5) # Default to 5 if not provided
                    
                    # Execute the local arXiv search function
                    tool_output = arxiv_search.invoke({"query": query, "max_results": max_results})
                        
                except SearchAPIError as error:
                    return _search_error_update(error)
                
                tool_messages.append(
                    ToolMessage(
                        content=str(tool_output),
                        tool_call_id=tool_call["id"],
                        name=tool_call["name"]
                    )
                )
                
    return {"messages": tool_messages}

## Define LangGraph Synthesis Node

Both search nodes feed `synthesize_results_node`, which turns the raw tool
output into a readable report: every retrieved record (title, link, excerpt or
abstract), followed by a reference list. When `citation_format_node` chose APA,
the list is built with the `apa_convert` tool; otherwise the retrieved
title/link citations are shown as-is.

`display_results_node` is the terminal node: it prints the synthesis produced by
`synthesize_results_node` and stores it in `state["answer"]`. A search node
whose API failed bypasses this section: its conditional edge sends the run
straight to `display_results_node`, which prints the readable error message.

In [ ]:
import json
import textwrap

from langchain_core.messages import AIMessage, ToolMessage

RULE = "=" * 72
THIN = "-" * 72


def _wrap(text: str, prefix: str = "", width: int = 88) -> str:
    """Collapse whitespace and wrap a paragraph under an aligned prefix."""
    return textwrap.fill(
        " ".join(str(text).split()),
        width=width,
        initial_indent=prefix,
        subsequent_indent=" " * len(prefix),
    )


def _field(label: str, value: str) -> str:
    """Render one aligned `Label: value` line."""
    return f"    {label + ':':<10}{value}"


def _parse_arxiv_block(block: str) -> dict:
    """Split one arxiv_search record into its labelled fields."""
    head, _, abstract = block.partition("Abstract:")
    fields = {}
    for line in head.splitlines():
        key, separator, value = line.partition(":")
        if separator:
            fields[key.strip().lower()] = value.strip()
    return {
        "title": fields.get("title") or "Untitled",
        "authors": fields.get("authors", ""),
        "published": fields.get("published", ""),
        "url": fields.get("url", ""),
        "abstract": " ".join(abstract.split()),
    }


def _format_tavily(content: str) -> str:
    """Render a Tavily result payload (JSON) as readable text."""
    data = json.loads(content)
    if not isinstance(data, dict) or ("results" not in data and "answer" not in data):
        raise ValueError("not a Tavily result payload")
    results = data.get("results", [])
    lines = [f"SOURCE: TAVILY_SEARCH  ({len(results)} web results)"]

    answer = " ".join(str(data.get("answer") or "").split())
    if answer:
        lines += ["", "Summary", _wrap(answer, prefix="  ")]

    for index, item in enumerate(results, 1):
        lines += ["", f"[{index}] {(item.get('title') or 'Untitled').strip()}"]
        if item.get("url"):
            lines.append(_field("Link", item["url"]))
        if item.get("content"):
            lines.append(_wrap(item["content"], prefix=f"    {'Excerpt:':<10}"))
    return "\n".join(lines)


def _format_arxiv(content: str) -> str:
    """Render the arxiv_search output (labelled text blocks) as readable text."""
    if "Title:" not in content:
        raise ValueError("not an arXiv result payload")
    blocks = [block.strip() for block in content.split("-" * 40) if block.strip()]
    lines = [f"SOURCE: ARXIV_SEARCH  ({len(blocks)} papers)"]

    for index, block in enumerate(blocks, 1):
        paper = _parse_arxiv_block(block)
        lines += ["", f"[{index}] {paper['title']}"]
        if paper["authors"]:
            lines.append(_wrap(paper["authors"], prefix=f"    {'Authors:':<10}"))
        if paper["published"]:
            lines.append(_field("Date", paper["published"]))
        if paper["url"]:
            lines.append(_field("Link", paper["url"]))
        if paper["abstract"]:
            lines.append(_wrap(paper["abstract"], prefix=f"    {'Abstract:':<10}"))
    return "\n".join(lines)


def _tavily_citations(content: str) -> list[str]:
    """Return the retrieved title/link pairs without APA reformatting."""
    data = json.loads(content)
    results = data.get("results", []) if isinstance(data, dict) else data
    citations = []
    for item in results:
        title = (item.get("title") or "").strip()
        url = (item.get("url") or "").strip()
        if title or url:
            citations.append(f"{title}. {url}".strip())
    return citations


def _arxiv_citations(content: str) -> list[str]:
    """Return the retrieved title/link pairs without APA reformatting."""
    if "Title:" not in content:
        raise ValueError("not an arXiv result payload")
    citations = []
    for block in content.split("-" * 40):
        paper = _parse_arxiv_block(block)
        if paper["title"] == "Untitled" and not paper["url"]:
            continue
        citations.append(f"{paper['title']}. {paper['url']}".strip())
    return citations


def synthesize_results_node(state: AgentState) -> dict:
    """
    Beautify the output of whichever search node ran.

    Renders the retrieved contents and always ends with a reference list: APA 7
    references via apa_convert() when chosen, otherwise the retrieved citations
    as-is.
    """
    question = state.get("question", "")
    citation_format = state.get("citation_format") or "as-is"
    sections = []
    reference_groups = []
    raw_citations = []

    messages = state.get("messages", [])
    # Only the current turn: in a persistent thread (LangGraph Studio) earlier
    # runs keep their ToolMessages, and their results must not be repeated.
    last_ai = next((m for m in reversed(messages) if isinstance(m, AIMessage)), None)
    current_tool_call_ids = {
        call.get("id") for call in (getattr(last_ai, "tool_calls", None) or [])
    }

    for message in messages:
        if not isinstance(message, ToolMessage):
            continue
        if message.name not in {"tavily_search", "arxiv_search"}:
            continue
        if current_tool_call_ids and message.tool_call_id not in current_tool_call_ids:
            continue

        content = message.content or ""
        try:
            if message.name == "tavily_search":
                sections.append(_format_tavily(content))
            else:
                sections.append(_format_arxiv(content))
        except Exception:
            # A failed search is a plain error string: show it as-is.
            sections.append(f"SOURCE: {message.name.upper()}  (no results)\n{content}")
            continue

        # Keep raw title/link pairs so citations are listed even when APA was
        # not requested (or when apa_convert cannot process the payload).
        try:
            if message.name == "tavily_search":
                raw_citations.extend(_tavily_citations(content))
            else:
                raw_citations.extend(_arxiv_citations(content))
        except Exception:
            pass

        if citation_format == "APA":
            try:
                reference_groups.append(apa_convert.invoke({"payload": content}))
            except ValueError:
                pass  # not citation data (already shown above)

    if not sections:
        return {"synthesis": "No search results were produced."}

    citation_note = "APA 7 (apa_convert)" if citation_format == "APA" else "as retrieved"
    lines = [
        RULE,
        "RESEARCH SYNTHESIS",
        RULE,
        f"Question: {question}",
        f"Citations: {citation_note}",
    ]
    for section in sections:
        lines += ["", section]

    if citation_format == "APA" and reference_groups:
        lines += ["", THIN, "REFERENCES (APA 7)", THIN]
        number = 0
        for group in reference_groups:
            for reference in group.split("\n\n"):
                reference = reference.strip()
                if reference:
                    number += 1
                    lines.append(f"[{number}] {reference}")
    elif raw_citations:
        lines += ["", THIN, "REFERENCES (AS RETRIEVED)", THIN]
        for number, citation in enumerate(raw_citations, 1):
            lines.append(f"[{number}] {citation}")

    return {"synthesis": "\n".join(lines)}


def display_results_node(state: AgentState) -> dict:
    """Terminal node: displays the run's result, then its Laya evaluation.

    The normal path prints the synthesis and its evaluation. A blocked request
    or a failed retrieval API prints the user-readable message instead. Keeping
    a single terminal node gives the graph exactly one END edge.
    """
    if state.get("threat_detected"):
        message = state.get("answer") or "Request blocked by the security guardrail."
        print(message)
        return {"answer": message}

    if state.get("api_error"):
        message = state["api_error"]
        print(message)
        return {"answer": message}

    synthesis = state.get("synthesis") or "No synthesis was produced."
    print(synthesis)

    eval_report = state.get("eval_report")
    if eval_report:
        print()
        print(eval_report)

    return {"answer": synthesis}


## Define LangGraph Question Node

`question_node` is the entry node. It reads the question from the initial
state (either `state["question"]` or the latest user `messages` entry), clears
the previous turn's results, and emits one `AIMessage` holding a tool call for
each search tool. `citation_format_node` then chooses the citation style,
the security node vets the input, and `research_decision_node` picks exactly one
search tool.


In [1]:
from uuid import uuid4

from langchain_core.messages import AIMessage

# The graph is run with the initial_state defined in the "Run The Graph"
# section, for example:
#   initial_state = {"messages": [("user", "your question here")]}
#   final_state = app.invoke(initial_state)


def question_node(state: AgentState) -> dict:
    """
    Entry node that normalizes the question.

    Takes the question from the state the graph is invoked with (initial_state)
    and turns it into a single AIMessage holding one tool call per search tool.
    Accepts either state["question"] or a user message in state["messages"].
    `citation_format_node` then chooses the citation style, the security node
    vets the input, and `research_decision_node` picks which search actually runs.
    """
    question = (state.get("question") or "").strip()

    if not question:
        for message in reversed(state.get("messages", [])):
            if isinstance(message, (tuple, list)):
                content = message[1] if len(message) > 1 else ""
            else:
                content = getattr(message, "content", "")
            question = str(content or "").strip()
            if question:
                break

    if not question:
        raise ValueError("initial_state must contain a 'question' or a user message.")

    tool_calls = [
        {
            "name": "tavily_search",
            "args": {"query": question},
            "id": f"tavily-{uuid4().hex[:8]}",
            "type": "tool_call",
        },
        {
            "name": "arxiv_search",
            "args": {"query": question, "max_results": 5},
            "id": f"arxiv-{uuid4().hex[:8]}",
            "type": "tool_call",
        },
    ]

    return {
        "question": question,
        # Clear per-run results so a persistent Studio thread never routes on
        # (or displays) an error or evaluation from an earlier turn.
        "api_error": "",
        "citation_format": "",
        "citation_scores": {},
        "citation_reason": "",
        "synthesis": "",
        "eval_score": None,
        "eval_report": "",
        "eval_details": {},
        "messages": [AIMessage(content=question, tool_calls=tool_calls)],
    }


## Define LangGraph Security Guardrail Node

A conditional node placed after `citation_format_node` and before
`research_decision_node`. It runs
`guardrails-ai-detect-prompt-injection` and `guardrails-ai-detect-jailbreak` on the
custom input. A flagged input raises `SecurityThreatError`, which the guardrail
node catches and routes to the terminal display node with a readable message;
a clean input goes to `research_decision_node`, which picks one search node.

**NOTE: This block will load guardrail weights.**

In [ ]:
import os
import warnings

# Guardrails ships an OpenTelemetry exporter that retries an unreachable collector;
# disabling tracing keeps the notebook output clean.
os.environ.setdefault("OTEL_SDK_DISABLED", "true")

# Guardrails prefers an asyncio loop; notebook cells run without one and fall
# back to synchronous validation, which is correct here. Hide the warning.
warnings.filterwarnings("ignore", message="Could not obtain an event loop.*")

from guardrails import Guard
from guardrails.settings import settings as guardrails_settings

guardrails_settings.disable_tracing = True

from langchain_core.messages import AIMessage
from guardrails_ai.detect_jailbreak import DetectJailbreak
from guardrails_ai.detect_prompt_injection import DetectPromptInjection
from langgraph.types import Command

# Security layer: both validators run on the custom input before any tool is called.
# Guard().use() stores one validator list per target, so both go in a single call.
# DetectJailbreak downloads its HuggingFace models the first time this cell runs.
# check_vector=False keeps the injection validator on its local heuristics: its
# optional embedding check would call OpenAI, and this notebook runs on Ollama
# with a placeholder OPENAI_API_KEY.
security_guard = Guard().use(
    DetectPromptInjection(check_vector=False, on_fail="noop"),
    DetectJailbreak(on_fail="noop"),
)

THREAT_MESSAGE = (
    "Request blocked: the input was flagged as a prompt-injection or jailbreak attempt, "
    "so no searches were run."
)


class SecurityThreatError(RuntimeError):
    """Raised when the guardrail flags the input; the message is user-readable."""

    def __init__(self, message: str = THREAT_MESSAGE):
        super().__init__(message)


def _validate_question(question: str) -> None:
    """Run the guardrails and raise SecurityThreatError for hostile input."""
    outcome = security_guard.validate(question)
    if not outcome.validation_passed:
        raise SecurityThreatError()


def security_check_node(state: AgentState) -> Command:
    """
    Conditional security node.

    Clean input is passed to the research decision node, which picks one search
    tool. A hostile input raises SecurityThreatError, which this node catches and
    routes to the terminal display node with a readable message, so no tool runs.
    """
    question = state.get("question", "")

    try:
        _validate_question(question)
    except SecurityThreatError as error:
        message = str(error)
        print(f"[security] {message}")
        return Command(
            goto="display_results_node",
            update={
                "threat_detected": True,
                "answer": message,
                "messages": [AIMessage(content=message)],
            },
        )

    return Command(
        goto="research_decision_node",
        update={"threat_detected": False},
    )


## Define LangGraph Research Decision Node

Classifies the question with Laya and routes it to exactly one retrieval tool:
`tavily_search_node` for web search or `arxiv_search_node` for academic
literature. Laya returns a calibrated probability for each option and both are
printed with the margin, so the choice can be audited.


In [ ]:
import os

import requests

# Laya (LAYA_* in .env) is a non-autoregressive decision model: instead of
# generating text it scores every option and returns calibrated probabilities,
# so the routing decision is a measured comparison rather than a guess.
laya_endpoint = os.getenv("LAYA_BASE_URL") or laya_base_url
if not laya_endpoint.startswith(("http://", "https://")):
    laya_endpoint = "http://" + laya_endpoint
laya_token = os.getenv("LAYA_API_KEY") or laya_api_key

# Routing rule: time-sensitive facts go to the web, research topics and papers
# go to arXiv. Laya caps each option at 48 tokens, so the criteria stay short.
ROUTE_OPTIONS = {
    "tavily_search_node": (
        "Time-sensitive information: recent events, announcements, current facts, news, prices, weather, sports, "
        "elections, releases, laws, regulations, current leaders and executives."
    ),
    "arxiv_search_node": (
        "Research papers and time-insensitive research topics: methods, theory, "
        "benchmarks, surveys, datasets, scientific explanations, paper definitions, prior search."
    ),
}

DECISION_QUESTION = {
    "route": {
        "type": "choice",
        "instructions": (
            "Which retrieval tool should answer? Use tavily for time-sensitive "
            "facts; use arxiv for research papers and time-insensitive research."
        ),
        "criteria": ROUTE_OPTIONS,
    }
}

_WEB_HINTS = (
    "news", "today", "latest", "now", "breaking", "price", "weather", "stock",
    "score", "election", "who is", "current", "trending", "release date",
)
_ACADEMIC_HINTS = (
    "paper", "arxiv", "study", "research", "benchmark", "survey", "theory",
    "dataset", "journal", "preprint", "literature", "state of the art",
)


def _ask_laya(question: str):
    """POST the routing question to Laya and return its calibrated probabilities."""
    headers = {"Content-Type": "application/json"}
    if laya_token:
        headers["Authorization"] = f"Bearer {laya_token}"

    response = requests.post(
        laya_endpoint,
        json={"state": {"question": question}, "questions": DECISION_QUESTION},
        headers=headers,
        timeout=30,
    )
    response.raise_for_status()
    probabilities = response.json()["answers"]["route"]["probabilities"]

    scores = {name: float(probabilities.get(name, 0.0)) for name in ROUTE_OPTIONS}
    total = sum(scores.values())
    if total <= 0:
        raise ValueError(f"Laya returned no usable probabilities: {probabilities!r}")
    return {name: value / total for name, value in scores.items()}


def _fallback_scores(question: str):
    """Keyword scoring, used only when Laya cannot be reached."""
    text = question.lower()
    web = sum(word in text for word in _WEB_HINTS)
    academic = sum(word in text for word in _ACADEMIC_HINTS)
    total = web + academic
    if total == 0:
        return {"tavily_search_node": 0.5, "arxiv_search_node": 0.5}
    web_score = web / total
    return {"tavily_search_node": web_score, "arxiv_search_node": 1 - web_score}


def research_decision_node(state: AgentState) -> Command:
    """
    Classifier node: Laya scores both retrieval tools and the higher score wins.

    Both calibrated probabilities and the margin are printed so the routing
    decision can be audited.
    """
    question = state.get("question", "")

    try:
        scores = _ask_laya(question)
        classifier = "Laya (calibrated probabilities)"
    except Exception as exc:  # Laya offline, timed out, or returned an error
        scores = _fallback_scores(question)
        classifier = "keyword fallback"
        print("[research_decision_node] WARNING: Laya unavailable, using keyword fallback")
        print("  error:", exc)

    web_score = scores["tavily_search_node"]
    academic_score = scores["arxiv_search_node"]
    route = "arxiv_search_node" if academic_score > web_score else "tavily_search_node"
    margin = abs(academic_score - web_score)

    print(f"[research_decision_node] classifier: {classifier}")
    print("  question:", question)
    print(f"  tavily_search_node   (web)      {web_score:6.3f}  {'#' * int(round(web_score * 40))}")
    print(f"  arxiv_search_node    (academic) {academic_score:6.3f}  {'#' * int(round(academic_score * 40))}")
    print(f"  -> chosen: {route}  (margin {margin:.3f})")

    return Command(
        goto=route,
        update={
            "research_route": route,
            "research_scores": scores,
            "research_reason": (
                f"{classifier}: {route} scored highest "
                f"(tavily={web_score:.3f}, arxiv={academic_score:.3f})"
            ),
        },
    )


## Define LangGraph Citation Format Node

`citation_format_node` sits between `question_node` and `security_node`. It asks
Laya whether the user wants APA-style citations, passing it the
citation-related sentence(s) so a long research question does not dilute an
explicit APA request. It prints both calibrated scores and the winning option,
and stores `citation_format` in the state. The synthesis
node reads that value: it always renders a reference list, using
`apa_convert` for APA and the retrieved title/link citations otherwise.


In [ ]:
import re

import requests

# The citation decision is a second Laya choice: does the user want APA style?
# It runs before the guardrail so the format is available to synthesis.
CITATION_OPTIONS = {
    "APA": (
        "The user wants APA-style citations: they explicitly ask for APA, a "
        "citation style, formatted references, or a bibliography."
    ),
    "as-is": (
        "The user did not ask for APA formatting; show the retrieved citations "
        "as they are."
    ),
}

CITATION_QUESTION = {
    "format": {
        "type": "choice",
        "instructions": "Does the user want citations in APA format?",
        "criteria": CITATION_OPTIONS,
    }
}

_APA_PATTERN = re.compile(r"\bapa\b|american psychological association", re.IGNORECASE)

_CITATION_PATTERN = re.compile(
    r"\b(?:apa|citations?|cite|cited|references?|bibliography)\b"
    r"|american psychological association",
    re.IGNORECASE,
)


def _citation_request(question: str) -> str:
    """Return the sentence(s) that ask about citation formatting, if any.

    A long research question can bury an explicit "please use APA" instruction.
    Laya judges better when it sees only the citation-related sentence(s).
    """
    sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", question) if s.strip()]
    matches = [
        sentence for sentence in sentences if _CITATION_PATTERN.search(sentence)
    ]
    return " ".join(matches) if matches else question


def _ask_laya_citation(question: str) -> dict:
    """POST the citation-format question to Laya and normalise its scores."""
    headers = {"Content-Type": "application/json"}
    if laya_token:
        headers["Authorization"] = f"Bearer {laya_token}"

    response = requests.post(
        laya_endpoint,
        json={"state": {"question": question}, "questions": CITATION_QUESTION},
        headers=headers,
        timeout=30,
    )
    response.raise_for_status()
    probabilities = response.json()["answers"]["format"]["probabilities"]

    scores = {name: float(probabilities.get(name, 0.0)) for name in CITATION_OPTIONS}
    total = sum(scores.values())
    if total <= 0:
        raise ValueError(f"Laya returned no usable probabilities: {probabilities!r}")
    return {name: value / total for name, value in scores.items()}


def _citation_fallback_scores(question: str) -> dict:
    """Keyword fallback, used only when Laya cannot be reached."""
    text = question.lower()
    apa = 1.0 if _APA_PATTERN.search(text) else 0.0
    return {"APA": apa, "as-is": 1.0 - apa}


def citation_format_node(state: AgentState) -> dict:
    """
    Decide whether the user wants APA citations, using Laya.

    Laya returns a calibrated probability for each option; both scores and the
    chosen format are printed and stored in the state for synthesis. Only the
    citation-related sentence(s) are sent to Laya, so a long research question
    cannot dilute an explicit APA request. A hostile question is still blocked
    by security_node later in the graph.
    """
    question = state.get("question", "")
    citation_request = _citation_request(question)

    try:
        scores = _ask_laya_citation(citation_request)
        classifier = "Laya (calibrated probabilities)"
    except Exception as exc:  # Laya offline, timed out, or returned an error
        scores = _citation_fallback_scores(citation_request)
        classifier = "keyword fallback"
        print("[citation_format_node] WARNING: Laya unavailable, using keyword fallback")
        print("  error:", exc)

    apa_score = scores["APA"]
    as_is_score = scores["as-is"]
    choice = "APA" if apa_score > as_is_score else "as-is"
    margin = abs(apa_score - as_is_score)

    print(f"[citation_format_node] classifier: {classifier}")
    print("  question:", question)
    if citation_request != question:
        print("  citation request:", citation_request)
    print(f"  APA       {apa_score:6.3f}  {'#' * int(round(apa_score * 40))}")
    print(f"  as-is     {as_is_score:6.3f}  {'#' * int(round(as_is_score * 40))}")
    print(f"  -> chosen: {choice}  (margin {margin:.3f})")

    return {
        "citation_format": choice,
        "citation_scores": scores,
        "citation_reason": (
            f"{classifier}: {choice} scored highest "
            f"(APA={apa_score:.3f}, as-is={as_is_score:.3f})"
        ),
    }


## Evaluate the Synthesis with Laya

Adds `eval_synthesis_node` between the synthesis and display nodes. Laya is a
System One decision model, so it judges one source/entry pair at a time rather
than the whole report at once: for each rendered entry it returns
`P(entry is stated in the source)`. The node aggregates those probabilities
into a 0-1 score, checks title/link coverage, and writes `eval_score`,
`eval_report` and `eval_details` to the state, which makes the result visible
in LangGraph Studio and in the terminal display.

Because the English checkpoint has a 512-token context, each comparison uses
the leading ~100 words of the source and entry; it is a fast screening judge,
not a substitute for a deliberative fact-check.

In [ ]:
import json
import os
import re

import requests
from langchain_core.messages import AIMessage, ToolMessage

# ---------------------------------------------------------------------------
# Laya synthesis evaluation
#
# Laya is a System One decision model: it scores short options against a short
# state. The report is therefore evaluated one source/entry pair at a time,
# rather than asking Laya to hold the whole report in its 512-token context.
# For each pair it answers: "Is the claim stated in the source?"
# ---------------------------------------------------------------------------

EVAL_ENTRY_CHECK = {
    "type": "choice",
    "instructions": "Is the claim stated in the source?",
    "criteria": {
        "A": "yes: the claim is directly stated in the source",
        "B": "no: the claim is absent from or contradicted by the source",
    },
}

EVAL_MAX_WORDS = 100
EVAL_PASS_THRESHOLD = 0.85
EVAL_REVIEW_THRESHOLD = 0.60


def _eval_endpoint():
    """Resolve Laya at call time; this cell can run before the Laya routing cell."""
    endpoint = os.getenv("LAYA_BASE_URL") or globals().get("laya_endpoint")
    if not endpoint:
        raise RuntimeError("LAYA_BASE_URL is not set")
    if not endpoint.startswith(("http://", "https://")):
        endpoint = "http://" + endpoint
    token = os.getenv("LAYA_API_KEY") or globals().get("laya_token")
    return endpoint, token


def _clip_words(text, limit=EVAL_MAX_WORDS):
    """Keep the leading words so each request stays inside Laya's state budget."""
    return " ".join(str(text or "").split()[:limit])


def _normalise(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text or "").lower()).strip()


def _current_turn_tool_messages(messages):
    """Only the ToolMessages produced for the latest AIMessage (one turn)."""
    last_ai = next((m for m in reversed(messages) if isinstance(m, AIMessage)), None)
    ids = {call.get("id") for call in (getattr(last_ai, "tool_calls", None) or [])}
    for message in messages:
        if not isinstance(message, ToolMessage):
            continue
        if message.name not in {"tavily_search", "arxiv_search"}:
            continue
        if ids and message.tool_call_id not in ids:
            continue
        yield message


def _collect_sources(messages):
    """Normalise Tavily and arXiv tool output into one source list per route."""
    sources = {"tavily_search": [], "arxiv_search": []}

    for message in _current_turn_tool_messages(messages):
        content = message.content or ""

        if message.name == "tavily_search":
            try:
                data = json.loads(content)
                results = data.get("results", []) if isinstance(data, dict) else data
            except (TypeError, ValueError):
                continue
            for item in results:
                if not isinstance(item, dict):
                    continue
                sources["tavily_search"].append({
                    "title": str(item.get("title") or "").strip(),
                    "url": str(item.get("url") or "").strip(),
                    "text": " ".join(str(item.get("content") or "").split()),
                })
        else:
            if "Title:" not in content:
                continue
            for block in content.split("-" * 40):
                head, _, abstract = block.partition("Abstract:")
                fields = {}
                for line in head.splitlines():
                    key, separator, value = line.partition(":")
                    if separator:
                        fields[key.strip().lower()] = value.strip()
                if not fields:
                    continue
                sources["arxiv_search"].append({
                    "title": fields.get("title", "").strip(),
                    "url": fields.get("url", "").strip(),
                    "text": " ".join(abstract.split()),
                })

    return sources


def _parse_synthesis_entries(synthesis):
    """Split the formatted report into entries, grouped by the search that produced them."""
    entries = {"tavily_search": [], "arxiv_search": []}
    route = None
    current = None

    for raw_line in synthesis.splitlines():
        line = raw_line.rstrip()

        if line.strip().startswith("REFERENCES"):
            route = None
            current = None
            continue

        match = re.match(r"^SOURCE: (TAVILY_SEARCH|ARXIV_SEARCH)", line)
        if match:
            route = "tavily_search" if match.group(1) == "TAVILY_SEARCH" else "arxiv_search"
            current = None
            continue

        match = re.match(r"^\[(\d+)\]\s*(.*)$", line.strip())
        if match and route:
            current = {
                "index": int(match.group(1)),
                "title": match.group(2).strip(),
                "lines": [],
            }
            entries[route].append(current)
            continue

        if current is not None:
            current["lines"].append(line)

    return entries


def _entry_fields(entry):
    """Extract the title, link and body (excerpt/abstract) of one report entry."""
    payload = "\n".join([entry["title"]] + entry["lines"])
    link_match = re.search(r"Link:\s*(\S+)", payload)
    body_match = re.search(r"(?:Excerpt|Abstract):\s*(.+)", payload, re.S)
    return {
        "index": entry["index"],
        "title": entry["title"],
        "url": link_match.group(1).strip() if link_match else "",
        "text": " ".join(body_match.group(1).split()) if body_match else "",
    }


def _laya_supports(source_text, claim_text):
    """Return P(the claim is stated in the source) from Laya."""
    endpoint, token = _eval_endpoint()
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = f"Bearer {token}"

    payload = {
        "state": {
            "source": _clip_words(source_text),
            "claim": _clip_words(claim_text),
        },
        "questions": {"supported": EVAL_ENTRY_CHECK},
    }
    response = requests.post(endpoint, json=payload, headers=headers, timeout=30)
    response.raise_for_status()
    answer = response.json()["answers"]["supported"]
    probabilities = answer.get("probabilities", {})
    yes = float(probabilities.get("A", 0.0))
    no = float(probabilities.get("B", 0.0))
    total = yes + no
    if total <= 0:
        raise ValueError(f"Laya returned no usable probabilities: {probabilities!r}")
    return yes / total


def eval_synthesis_node(state: AgentState) -> dict:
    """
    Judge the formatted synthesis against the sources it was built from.

    Laya compares each source/entry pair (short inputs, one forward pass each)
    and returns P(the entry is stated in the source). The node adds a 0-1 score,
    a readable report for display_results_node, and structured details for
    LangGraph Studio.
    """
    synthesis = state.get("synthesis") or ""
    sources = _collect_sources(state.get("messages", []))
    entries = _parse_synthesis_entries(synthesis)

    total_sources = sum(len(items) for items in sources.values())
    total_entries = sum(len(items) for items in entries.values())

    if not synthesis or total_entries == 0:
        details = {"error": "No synthesis entries to evaluate."}
        return {
            "eval_score": None,
            "eval_details": details,
            "eval_report": (
                "SYNTHESIS EVALUATION (Laya)\n"
                "Evaluation skipped: no synthesis entries."
            ),
        }

    checks = []
    matched = 0
    errors = []

    for route in ("tavily_search", "arxiv_search"):
        source_items = sources.get(route, [])
        entry_items = entries.get(route, [])

        for position, entry in enumerate(entry_items):
            entry_fields = _entry_fields(entry)
            source = source_items[position] if position < len(source_items) else None

            if source is None:
                checks.append({
                    "route": route,
                    "index": entry_fields["index"],
                    "title": entry_fields["title"] or "Untitled",
                    "p_supported": 0.0,
                    "metadata_ok": False,
                    "status": "no matching source",
                })
                continue

            matched += 1
            title_ok = _normalise(entry_fields["title"]) == _normalise(source["title"])
            url_ok = entry_fields["url"].rstrip("/") == source["url"].rstrip("/")
            metadata_ok = title_ok and url_ok

            if not entry_fields["text"] or not source["text"]:
                # Nothing textual to compare; fall back to the metadata check.
                p_supported = 1.0 if metadata_ok else 0.0
                status = "metadata only"
            else:
                try:
                    p_supported = _laya_supports(source["text"], entry_fields["text"])
                    status = "supported" if p_supported >= 0.5 else "unsupported"
                except Exception as exc:
                    errors.append(f"{route}[{position + 1}]: {exc}")
                    p_supported = None
                    status = "error"

            checks.append({
                "route": route,
                "index": entry_fields["index"],
                "title": entry_fields["title"] or "Untitled",
                "p_supported": p_supported,
                "metadata_ok": metadata_ok,
                "status": status,
            })

    judged = [check for check in checks if check["p_supported"] is not None]
    if not judged:
        error = "; ".join(errors) or "Laya did not return an evaluation."
        details = {"error": error, "checks": checks}
        return {
            "eval_score": None,
            "eval_details": details,
            "eval_report": (
                "SYNTHESIS EVALUATION (Laya)\n"
                f"Evaluation failed: {error}"
            ),
        }

    grounding = sum(
        check["p_supported"] * (1.0 if check["metadata_ok"] else 0.5)
        for check in judged
    ) / len(judged)
    coverage = matched / max(total_sources, total_entries, 1)
    score = round(0.85 * grounding + 0.15 * coverage, 3)

    if score >= EVAL_PASS_THRESHOLD and not errors:
        verdict = "PASS"
    elif score >= EVAL_REVIEW_THRESHOLD:
        verdict = "REVIEW"
    else:
        verdict = "FAIL"

    details = {
        "evaluator": "Laya",
        "score": score,
        "verdict": verdict,
        "grounding": round(grounding, 3),
        "coverage": round(coverage, 3),
        "sources": total_sources,
        "entries": total_entries,
        "judged": len(judged),
        "checks": checks,
        "errors": errors,
    }

    lines = [
        "=" * 72,
        "SYNTHESIS EVALUATION (Laya)",
        "=" * 72,
        f"Route:            {state.get('research_route', 'unknown')}",
        f"Sources/entries:  {total_sources}/{total_entries}",
        "Judge:            Laya (P(entry supported by source))",
        f"Grounding:        {grounding:.3f}",
        f"Coverage:         {coverage:.3f}",
        f"Eval score:       {score:.3f}  [{verdict}]",
        "",
        "Per-entry checks:",
    ]
    for check in checks:
        p_value = check["p_supported"]
        p_text = "n/a" if p_value is None else f"{p_value:.3f}"
        metadata = "ok" if check["metadata_ok"] else "metadata mismatch"
        lines.append(
            f"  [{check['index']}] {p_text}  {check['status']:<16} "
            f"{metadata}  {check['title'][:56]}"
        )
    if errors:
        lines += ["", "Errors:"] + [f"  - {error}" for error in errors]

    return {
        "eval_score": score,
        "eval_details": details,
        "eval_report": "\n".join(lines),
    }


## Assemble the Graph
When using LangGraph Studio, the studio interface or API can be used to enter the message.  

`GRAPH_CONFIG` caps a run at 10 supersteps (`recursion_limit`). LangGraph Studio
gets the same limit from `LANGGRAPH_CONFIG`, set by
`run_langgraph_studio.bat`.

In [ ]:
from langgraph.graph import StateGraph, END, START


# LangGraph Studio / the API only need a question (or messages) to start a
# run. Declaring the input schema separately keeps the other AgentState
# fields out of the Studio input form while leaving the shared state intact.
class AgentInput(TypedDict, total=False):
    question: str
    messages: Annotated[list, add_messages]


graph = StateGraph(AgentState, input_schema=AgentInput)

# Nodes
graph.add_node("question_node", question_node)
graph.add_node("citation_format_node", citation_format_node)
graph.add_node(
    "security_node",
    security_check_node,
    # Command(goto=...) is invisible to the graph drawing, so declare the
    # possible destinations for Studio. Rendering only; execution still
    # follows the Command returned by the node.
    destinations={"research_decision_node": "clean", "display_results_node": "threat"},
)
graph.add_node(
    "research_decision_node",
    research_decision_node,
    destinations={"tavily_search_node": "web", "arxiv_search_node": "papers"},
)
graph.add_node("tavily_search_node", tavily_search_node)
graph.add_node("arxiv_search_node", arxiv_search_node)
graph.add_node("synthesize_results_node", synthesize_results_node)
graph.add_node("eval_synthesis_node", eval_synthesis_node)
graph.add_node("display_results_node", display_results_node)

# Entry edge: START hands the initial state to the question node
graph.add_edge(START, "question_node")

# citation_format_node decides the citation style, then security vets the input.
graph.add_edge("question_node", "citation_format_node")
graph.add_edge("citation_format_node", "security_node")

# security_check_node returns a Command: a threat routes to display_results_node
# (the node caught SecurityThreatError), a clean input goes to
# "research_decision_node". research_decision_node returns a Command picking
# exactly one search node. Every branch ends at display_results_node, so the
# graph has a single terminal node and therefore a single END edge.

# Both search nodes feed the same synthesis node, which formats the results
# (contents + APA citations) before the terminal display node prints them.
# If the retrieval API raised SearchAPIError, the search node stored a readable
# message and the conditional edge skips synthesis/evaluation entirely.
def _route_after_search(state: AgentState) -> str:
    """Send a failed retrieval API call straight to the terminal display."""
    if state.get("api_error"):
        return "api_error"
    return "results"


graph.add_conditional_edges(
    "tavily_search_node",
    _route_after_search,
    {"results": "synthesize_results_node", "api_error": "display_results_node"},
)
graph.add_conditional_edges(
    "arxiv_search_node",
    _route_after_search,
    {"results": "synthesize_results_node", "api_error": "display_results_node"},
)
graph.add_edge("synthesize_results_node", "eval_synthesis_node")
graph.add_edge("eval_synthesis_node", "display_results_node")
graph.add_edge("display_results_node", END)

# Graph-level run configuration. LangGraph defaults the recursion limit to 25;
# this pipeline needs 8 supersteps, so 10 caps a runaway loop early.
GRAPH_CONFIG = {"recursion_limit": 10}

app = graph.compile()


## Run The Graph
You may specify the message here if you do not intend to do it via LangGraph Studio. 

In [ ]:
from langchain_core.runnables import RunnableLambda, RunnableParallel
print("============================")
print("THE RESEARCH ASSISTANT GRAPH")
print("============================")
print("\n")
app.get_graph().print_ascii()
print("\n\n")

# SCENARIO 1: The Travily search node is chosen by Laya, and it returns a few web results. The synthesis is produced and evaluated.
print("===================================")
print("SCENARIO 1: The Travily Search Path")
print("===================================")
initial_state = {"messages": [("user", "What is the latest news about the global economy?")]}
final_state = app.invoke(initial_state, config=GRAPH_CONFIG)

# SCENARIO 2: The Travily search node is chosen by Laya, and it returns a few web results. The synthesis is produced and evaluated.
# The user explicitly asks for APA citations, so the synthesis is formatted accordingly.
print("===================================")
print("SCENARIO 2: The Travily Search Path (APA citations)")
print("===================================")
initial_state = {"messages": [("user", "What is the latest news about the global economy? Please provide APA-style citations.")]}
final_state = app.invoke(initial_state, config=GRAPH_CONFIG)

# SCENARIO 3: The arXiv search node is chosen by Laya, and it returns a few papers. The synthesis is produced and evaluated.
print("\n\n")
print("===================================")
print("SCENARIO 3: The arXiv Search Path")
print("===================================")
initial_state = {"messages": [("user", "Recent advances in retrieval-augmented generation for scientific literature.")]}
final_state = app.invoke(initial_state, config=GRAPH_CONFIG)

# display_results_node already printed the synthesis and its evaluation during
# the run, or the block/API-error message when there was no result to show.
